# 14 · Modelos con y sin régimen: walk-forward

En los notebooks 11–13 examinamos modelos con CPCV. Ahora construiremos una referencia que respete el orden cronológico: **aprender del pasado y predecir el año siguiente**. Primero construiremos la referencia y después mediremos qué aporta añadir información del entorno.

- **M0:** frecuencia de éxitos del entrenamiento de cada año.
- **M1:** logística L2 con retorno del último mes y momentum 12–1.
- **M2:** la misma logística, añadiendo volatilidad, retorno de 24 meses y drawdown de 12 meses.
- **M3:** M2 más la interacción mercado bajista × volatilidad.

«Sin régimen» es una definición operativa: estas dos variables describen las señales y también pueden reflejar indirectamente el mercado. M2 añade las tres variables directas de régimen. M3 estudia la interacción prevista, después se examinan las tres retiradas individuales de M2.

**Entradas:** tablas de 01 → 02 → 07 → 08 y el entorno `.venv` con el extra `models`. La sección CPCV necesita además las tablas del notebook 10, la parte walk-forward es independiente de ellas. No necesitas repetir el 09 ni 11–13. El diseño, motivado por la guía del 15 de noviembre, está en [el protocolo](../docs/validacion_walk_forward.md). Se ha definido después de conocer los experimentos anteriores.
La última parte evalúa los mismos seis candidatos con CPCV como robustez. Conserva su propio resumen y muestra también 1950–1989 para alinear los meses objetivo con la prueba cronológica.

Las secciones 19–22 traducen las predicciones cronológicas a cuentas continuas con umbral fijo mayor que 0,5, y las comparan con las referencias económicas. Pueden ejecutarse por separado sin recalcular modelos.

Las secciones 23–25 incorporan random forest con los mismos candidatos y calendario, sin volver a ajustar la logística. Sus comprobaciones se mantienen separadas de las ya realizadas para el modelo lineal.


## 1. Un calendario que avanza año a año

El desarrollo sigue siendo 1927–1989. Para disponer de historia inicial, empezamos las pruebas en 1950. Aprendemos con la historia anterior y mantenemos los coeficientes fijos durante los doce meses de ese año. Al año siguiente ampliamos el entrenamiento y volvemos a ajustar. Son cuarenta periodos anuales de prueba, hasta 1989.

Aplicamos una purga de un mes objetivo anterior. Para probar 1950 usamos etiquetas hasta noviembre de 1949, diciembre de 1949 queda excluido. El calendario se define antes de retirar meses sin señal o sin características. Los meses futuros nunca entrenan ese ajuste. Como no hay entrenamiento posterior a su prueba, el embargo posterior de CPCV no elimina meses aquí.

La normalización causal de las características sí se actualiza con la historia anterior a cada nueva observación. Mantener los coeficientes durante el año no implica congelar en enero esa preparación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import display

from tfm.data import project_root, save_result
from tfm.walk_forward import (FIRST_TEST_YEAR, LAST_TEST_YEAR, SIGNAL_COLUMNS,
    build_walk_forward_splits, load_walk_forward_inputs, summarize_walk_forward, run_walk_forward)

ROOT = project_root()
ejemplos, _, _ = load_walk_forward_inputs(ROOT)
particiones = build_walk_forward_splits()
cobertura = summarize_walk_forward(ejemplos, particiones)
display(cobertura.loc[cobertura.test_year.isin([FIRST_TEST_YEAR, LAST_TEST_YEAR])])

colores = {'train': ('Entrenamiento', '#9dc8b4'), 'purge': ('Purga', '#b74e40'),
           'validation': ('Prueba del año', '#244b70'), 'future': ('Futuro excluido', '#eeeeee')}
matriz = particiones.pivot(index='test_year', columns='target_period', values='role')
codigos = {nombre: i for i, nombre in enumerate(colores)}
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(matriz.replace(codigos).to_numpy(dtype=int), aspect='auto', interpolation='nearest',
          cmap=ListedColormap([color for _, color in colores.values()]), vmin=-.5, vmax=3.5)
x = [i for i, mes in enumerate(matriz.columns) if mes.endswith('-01') and int(mes[:4]) % 10 == 0]
y = list(range(0, len(matriz), 5))
ax.set_xticks(x, [matriz.columns[i][:4] for i in x])
ax.set_yticks(y, [matriz.index[i] for i in y])
ax.set(title='Historia disponible para cada año de prueba', xlabel='Mes objetivo del calendario', ylabel='Año de prueba')
fig.legend(handles=[Patch(color=color, label=nombre) for nombre, color in colores.values()],
           loc='lower center', ncol=4, fontsize=9)
fig.tight_layout(rect=(0, .06, 1, 1))
plt.show()

## 2. Qué ejemplos usamos y qué aprende M1

Una etiqueta indica si la operación aislada de la regla, neta de costes y dividendos aproximados, superó al efectivo. Solo tienen etiqueta las señales activas. Un mes sin operación no es un fallo del filtro y no recibe probabilidad cero.

Conservamos la muestra común de los experimentos anteriores: las diez características preparadas deben estar disponibles tras su calentamiento. M1 solo recibe **dos columnas explícitas**: `z_price_return_1m` y `z_momentum_12_1`. Las restantes no se utilizan para predecir. Así las comparaciones posteriores tendrán las mismas fechas, en lugar de ampliar la muestra de cada modelo por separado.

Usamos la misma logística regularizada del 11, con C=1, sin ponderar clases ni buscar parámetros. Las dos reglas se ajustan por separado. Los años sin señales permanecen en la cobertura. Si solo hay una clase de prueba, la log-loss se puede calcular y el AUC no.

In [ ]:
print('Predictores de M1:', SIGNAL_COLUMNS)
display(cobertura.groupby('strategy').agg(
    ejemplos_iniciales=('train_examples', 'min'),
    ejemplos_prueba=('validation_examples', 'sum'),
    minimo_anual=('validation_examples', 'min'),
    maximo_anual=('validation_examples', 'max')))
display(cobertura.loc[cobertura.validation_classes.lt(2),
    ['test_year', 'strategy', 'validation_calendar_months', 'no_signal_months',
     'excluded_active_months', 'validation_examples', 'validation_classes']])

## 3. Ajustar con el pasado y predecir una sola vez

M0 asigna a cada ejemplo del año la proporción de etiquetas positivas de su entrenamiento. M1 estima una probabilidad a partir de las dos características. Ninguno utiliza las etiquetas del año probado para ajustarse.

A diferencia del resumen de CPCV, cada ejemplo recibe **una única predicción cronológica** de cada modelo. No se promedian cinco modelos ni se cuentan repetidamente los mismos resultados. Si un ajuste no es estimable, sus probabilidades quedan ausentes, no se eliminan esos meses para mejorar el resumen.

In [ ]:
def mostrar_progreso(completadas, total):
    if completadas == 1 or completadas % 5 == 0:
        print(f'Años evaluados: {completadas}/{total}', flush=True)

tablas, informe = run_walk_forward(ROOT, progress=mostrar_progreso)
resumen = tablas['walk_forward_summary']
predicciones = tablas['walk_forward_predictions']
metricas = tablas['walk_forward_metrics']
coeficientes = tablas['walk_forward_coefficients']
print('Estado del recorrido:', informe['experiment_status'])
display(resumen[['test_year', 'strategy', 'train_first_target', 'train_last_target',
                'train_examples', 'validation_examples', 'train_classes', 'validation_classes', 'status']])

## 4. Leer los resultados

**AUC** describe si las etiquetas favorables reciben probabilidades más altas que las desfavorables. **Log-loss** penaliza probabilidades equivocadas, especialmente las muy seguras. Menor log-loss es favorable, mayor AUC indica mejor ordenación. Ninguna mide por sí sola rentabilidad.

El resumen principal (`pooled`) reúne las predicciones de prueba de los cuarenta años por regla y calcula la métrica una sola vez. No es una media de las AUC anuales. M0 tiene una probabilidad constante dentro de cada año, pero puede cambiar al reentrenarse: su AUC reunida puede diferir de 0,5.

Mostramos también cada año. `single_class` significa que no podemos calcular AUC porque todas las etiquetas son iguales, `no_examples`, que no hubo ejemplos. Estos años permanecen en las tablas. Si faltase una predicción necesaria, el resumen correspondiente quedaría no disponible.

In [ ]:
display(metricas.loc[metricas.aggregation.eq('pooled'),
    ['strategy', 'model', 'examples', 'positive', 'negative', 'log_loss', 'auc', 'status', 'auc_status']])
anuales = metricas.loc[metricas.aggregation.eq('year')]
display(anuales[['test_year', 'strategy', 'model', 'examples', 'log_loss', 'auc', 'status', 'auc_status']])
display(anuales.loc[anuales.model.eq('M1')].groupby(['strategy', 'auc_status']).size().rename('Años').reset_index())

## 5. Ver las probabilidades a lo largo del tiempo

Las líneas muestran las estimaciones de M0 y M1. Los puntos permiten ver también los meses aislados. Los huecos son meses sin ejemplo utilizable: no los rellenamos ni conectamos mediante interpolación. Una probabilidad alta significa que el modelo considera más probable superar al efectivo según nuestra etiqueta, no representa el tamaño de una ganancia ni una recomendación de inversión.

In [ ]:
import matplotlib.dates as mdates

meses_prueba = pd.period_range(f'{FIRST_TEST_YEAR}-01', f'{LAST_TEST_YEAR}-12', freq='M').astype(str)
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, regla in zip(axes, ('momentum', 'reversal')):
    serie = predicciones.loc[predicciones.strategy.eq(regla)].set_index('target_period').reindex(meses_prueba)
    fechas = pd.PeriodIndex(serie.index, freq='M').to_timestamp()
    ax.plot(fechas, serie.probability_naive, label='M0 · frecuencia pasada', color='#c77832', linewidth=1.2, marker='.', markersize=2)
    ax.plot(fechas, serie.probability_logistic, label='M1 · información de las señales', color='#244b70', linewidth=1, marker='.', markersize=2)
    ax.set(title=regla, ylabel='Probabilidad estimada', ylim=(0, 1))
    ax.grid(alpha=.2)
axes[0].legend(loc='upper left', fontsize=8)
axes[-1].set_xlabel('Mes objetivo de prueba')
axes[-1].xaxis.set_major_locator(mdates.YearLocator(10))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.tight_layout()
plt.show()

## 6. Guardar la referencia M0/M1

Esta primera parte establece y guarda la referencia para estudiar qué aporta el entorno. La sección siguiente añade M2 con las mismas fechas, ejemplos y parámetros. No seleccionamos variables ni umbrales de operación. Las sombras de la logística de diez características no validan automáticamente estas variantes.

El orden temporal evita entrenar con resultados posteriores al año de prueba. Sin embargo, seguimos usando precios medios mensuales de Shiller, dividendos interpolados y versiones actuales de las fuentes. La purga no certifica sus fechas históricas de publicación. Es una aproximación retrospectiva, no acredita operaciones ejecutables ni disponibilidad point-in-time. El periodo desde 1990 permanece reservado.

La siguiente celda guarda las cinco tablas de referencia con sus JSON. Es necesario publicarlas antes de ejecutar M2. Repetir su publicación retira las comparaciones de régimen anteriores, que deberán recalcularse. El historial del intento está en `outputs/experiments/walk_forward_trials.jsonl`. Si falla el guardado, repite la celda completa. Antes del commit, limpia las salidas y guarda.

Referencia: guía del tutor del 15 de noviembre de 2026, apartado 5, pp. 19–21. La adaptación, el calendario y los supuestos están en [el protocolo](../docs/validacion_walk_forward.md). Se reutilizan las metodologías y referencias de preparación, etiquetado y logística ya documentadas en el proyecto.

In [ ]:
guardados = []
for nombre, tabla in tablas.items():
    salida = save_result(ROOT, nombre, tabla, informe)
    guardados.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados))
print('Historial:', informe['trial_log'])
print('Evaluación pendiente:', informe['skill_gate_status'])

## 7. Qué información añade M2

Ahora planteamos una comparación concreta: ¿la logística mejora al conocer también las condiciones del mercado? M2 conserva las dos entradas de M1 y añade tres variables ya construidas en el notebook 07:

- **Volatilidad de 12 meses:** cuánto han variado los retornos recientes.
- **Retorno de 24 meses:** si el mercado venía subiendo o cayendo y con qué magnitud.
- **Drawdown de 12 meses:** distancia del precio al máximo de esa ventana.

Puede verse como comparar dos decisiones con el mismo historial de señales: una dispone de esa información básica y la otra conoce también el entorno. Todo lo demás permanece igual: logística L2 C=1, ejemplos, etiquetas, años, purga y transformación causal.

Las cinco columnas están fijadas por el protocolo. No se escogen según el resultado de M2, no se usa todavía Clustered MDA y la interacción mercado bajista × volatilidad se estudia por separado en la sección de M3. Esta comparación mide la aportación del **conjunto de tres variables**, no demuestra qué parte corresponde a cada una.

In [ ]:
from tfm.walk_forward import REGIME_COLUMNS, M2_COLUMNS, run_walk_forward_regime

print('M1 · información de las señales:', SIGNAL_COLUMNS)
print('M2 · variables de régimen añadidas:', REGIME_COLUMNS)
print('Total de predictores de M2:', len(M2_COLUMNS))

tablas_regimen, informe_regimen = run_walk_forward_regime(ROOT, progress=mostrar_progreso)
comparacion_regimen = tablas_regimen['walk_forward_regime_comparison']
metricas_regimen = tablas_regimen['walk_forward_regime_metrics']
predicciones_regimen = tablas_regimen['walk_forward_regime_predictions']
print('Estado del recorrido M2:', informe_regimen['experiment_status'])
display(pd.DataFrame(informe_regimen['diagnostics'])[
    ['test_year', 'strategy', 'train_examples', 'validation_examples', 'train_positive', 'iterations', 'status']])

## 8. Comparar los mismos ejemplos

La tabla calcula **AUC de M2 menos AUC de M1**. Una diferencia positiva indica que añadir el régimen mejora la ordenación de las etiquetas. También calcula **log-loss de M1 menos log-loss de M2**, para que una cifra positiva indique mejora en ambos casos. Son diferencias de métricas, no porcentajes de rentabilidad.

La comparación principal reúne una predicción de cada modelo por ejemplo. Por separado, conservamos todos los años. Si en un año hay una sola clase, no se puede medir su mejora de AUC, aunque sí su diferencia de log-loss. Si falta una predicción necesaria, la comparación correspondiente queda no disponible.

La estabilidad exige una mayoría estricta de los cuarenta años previstos: al menos veintiuno con mejora de AUC. Los años sin AUC estimable permanecen en ese total y no se cuentan como favorables. Se muestra también cuántos pudieron evaluarse. Estos años no son ensayos independientes y algunos contienen pocos ejemplos.
En el gráfico, verde indica mejora y rojo empeoramiento. Una diferencia exactamente cero no produce una barra visible, la cruz gris identifica los años cuyo AUC no puede calcularse.


In [ ]:
from matplotlib.lines import Line2D

columnas_comparacion = ['strategy', 'examples', 'auc_m1', 'auc_m2', 'auc_improvement',
    'log_loss_m1', 'log_loss_m2', 'log_loss_improvement', 'comparison_status']
display(comparacion_regimen.loc[comparacion_regimen.aggregation.eq('pooled'), columnas_comparacion])
display(pd.DataFrame(informe_regimen['stability']))
display(comparacion_regimen.loc[comparacion_regimen.aggregation.eq('year'),
    ['test_year', *columnas_comparacion, 'auc_improved']])

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, regla in zip(axes, ('momentum', 'reversal')):
    datos = comparacion_regimen.loc[comparacion_regimen.aggregation.eq('year') & comparacion_regimen.strategy.eq(regla)]
    cambio = datos.auc_improvement.to_numpy(dtype=float)
    ax.bar(datos.test_year.to_numpy(dtype=int), cambio,
           color=np.where(cambio > 0, '#43876b', '#b65b49'), label='Diferencia de AUC')
    sin_auc = datos.loc[datos.auc_improvement.isna(), 'test_year'].to_numpy(dtype=int)
    ax.scatter(sin_auc, np.zeros(len(sin_auc)), marker='x', color='#777777', s=22, label='AUC no estimable')
    ax.axhline(0, color='black', linewidth=.8)
    ax.set(title=regla, ylabel='AUC M2 − AUC M1')
    ax.grid(axis='y', alpha=.2)
axes[0].legend(handles=[Patch(color='#43876b', label='Mejora de AUC'),
    Patch(color='#b65b49', label='Empeoramiento de AUC'),
    Line2D([0], [0], marker='x', color='#777777', linestyle='None', label='AUC no estimable')], fontsize=8)
axes[-1].set_xlabel('Año de prueba')
fig.tight_layout()
plt.show()

## 9. Qué podemos concluir en este punto

Una mejora global permite describir una aportación predictiva provisional del bloque de régimen. Su aceptación requiere todavía las comprobaciones previstas: comparación equivalente en CPCV, sombras específicas, estabilidad y Sharpe neto frente al modelo sin régimen y a las referencias. Tampoco una mejora de clasificación garantiza que las operaciones aceptadas ganen más dinero.

Si M2 empeora o no mejora en suficientes años, se documenta el resultado sin cambiar ventanas, variables o parámetros para favorecerlo. A continuación se estudia M3 con la interacción ya prevista. Las retiradas individuales se muestran en la última parte. La selección interna y la evaluación económica siguen pendientes. El periodo desde 1990 permanece reservado. Siguen vigentes las limitaciones de precios medios mensuales y fuentes reconstruidas.

La siguiente celda guarda cuatro tablas adicionales, vinculadas al intento concreto de M0/M1 que se ha leído. M2 solo añade un candidato por regla, los cuarenta años no son cuarenta configuraciones. Su historial comparte `outputs/experiments/walk_forward_trials.jsonl`, distinguiendo las variantes. Si falla la publicación, repite la celda completa.

In [ ]:
guardados_regimen = []
for nombre, tabla in tablas_regimen.items():
    salida = save_result(ROOT, nombre, tabla, informe_regimen)
    guardados_regimen.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados_regimen))
print('Historial:', informe_regimen['trial_log'])
print('Validación de M2 pendiente:', informe_regimen['skill_gate_status'])

## 10. M3: ¿la volatilidad actúa igual después de una caída?

M2 permite que volatilidad y estado del mercado aporten efectos separados. M3 añade la interacción previamente definida: **indicador(retorno de 24 meses < 0) × volatilidad de 12 meses**. Así el modelo puede representar una relación con la volatilidad distinta en un entorno bajista.

La calculamos con los valores originales, antes de normalizarlos. Si el retorno previo de dos años es −10 % y la volatilidad anualizada es 20 %, el producto es 0,20. Si el retorno es positivo o exactamente cero, el producto vale cero. No utilizamos el signo del z-score del retorno: estar por debajo de su media histórica no significa necesariamente haber caído.

Después aplicamos la transformación causal ya usada en el proyecto: recorte a percentiles 1/99 y z-score con al menos 36 meses anteriores. Se calcula sobre el calendario completo, antes de seleccionar las señales de cada regla. La misma interacción acompaña a ambas reglas.

Al principio no hay ningún mercado bajista según este indicador y todo su pasado vale cero. Los límites de recorte son entonces cero y el valor recortado también. Usamos la convención de columna constante: contribución normalizada cero, sin dividir por cero ni inventar una desviación. Esto incluye la primera observación positiva si el recorte causal todavía la lleva a cero. El calentamiento mínimo sigue vigente y los datos ausentes no se rellenan. La misma convención se aplica si los percentiles causales vuelven a recortar todo el pasado a cero. Cuando hay variación histórica después del recorte, se utiliza el z-score habitual. Esta convención se fija antes de ajustar M3 y mantiene la misma muestra de entrenamiento que M1/M2.

In [ ]:
from tfm.data import read_result
from tfm.walk_forward import (INTERACTION_COLUMN, M3_COLUMNS,
    build_interaction_feature, run_walk_forward_interaction)

caracteristicas, _ = read_result(ROOT, 'monthly_features')
interaccion = build_interaction_feature(caracteristicas)
print('Predictores de M3:', M3_COLUMNS)
display(interaccion.loc[interaccion.target_period.between('1930-08', '1931-02')])
print('Meses con convención de historia cero:', int(interaccion.scaling_status.eq('constant_zero_encoded').sum()))

tablas_interaccion, informe_interaccion = run_walk_forward_interaction(ROOT, progress=mostrar_progreso)
comparacion_interaccion = tablas_interaccion['walk_forward_interaction_comparison']
print('Estado del recorrido M3:', informe_interaccion['experiment_status'])

## 11. Separar la aportación de la interacción de la del bloque completo

**M3 frente a M2** mide qué cambia al añadir la interacción. **M3 frente a M1** compara el conjunto completo con la referencia sin régimen explícito. Son dos lecturas del mismo candidato M3, no dos entrenamientos diferentes ni una oportunidad para escoger la comparación más favorable.

Se mantienen los mismos ejemplos, años, parámetros y criterio de estabilidad. La diferencia de AUC es M3 menos la referencia indicada, la mejora de log-loss es la referencia menos M3. Una cifra positiva es favorable. Los empates y años sin AUC estimable se conservan, sin contarlos como mejoras.

El gráfico muestra la comparación más directa, M3 frente a M2. La tabla muestra también M3 frente a M1. Ninguna comparación sustituye a las sombras específicas, la robustez CPCV o la evaluación económica.

In [ ]:
columnas_interaccion = ['reference_model', 'strategy', 'examples', 'auc_reference', 'auc_m3',
    'auc_improvement', 'log_loss_reference', 'log_loss_m3', 'log_loss_improvement', 'comparison_status']
display(comparacion_interaccion.loc[comparacion_interaccion.aggregation.eq('pooled'), columnas_interaccion])
display(pd.DataFrame(informe_interaccion['stability']))
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, regla in zip(axes, ('momentum', 'reversal')):
    datos = comparacion_interaccion.loc[comparacion_interaccion.aggregation.eq('year') &
        comparacion_interaccion.reference_model.eq('M2') & comparacion_interaccion.strategy.eq(regla)]
    cambio = datos.auc_improvement.to_numpy(dtype=float)
    ax.bar(datos.test_year.to_numpy(dtype=int), cambio, color=np.where(cambio > 0, '#43876b', '#b65b49'))
    sin_auc = datos.loc[datos.auc_improvement.isna(), 'test_year'].to_numpy(dtype=int)
    ax.scatter(sin_auc, np.zeros(len(sin_auc)), marker='x', color='#777777', s=22)
    ax.axhline(0, color='black', linewidth=.8)
    ax.set(title=regla, ylabel='AUC M3 − AUC M2')
    ax.grid(axis='y', alpha=.2)
axes[0].legend(handles=[Patch(color='#43876b', label='Mejora de AUC'),
    Patch(color='#b65b49', label='Empeoramiento de AUC'),
    Line2D([0], [0], marker='x', color='#777777', linestyle='None', label='AUC no estimable')], fontsize=8)
axes[-1].set_xlabel('Año de prueba')
fig.tight_layout()
plt.show()

## 12. Guardar M3 y conservar las referencias

La interacción es un candidato previsto antes de ver M2, no se ha elegido probando distintas fórmulas tras sus resultados. Su tratamiento de la historia constante se ha fijado al comprobar cobertura y antes de ajustar M3. Las diez características originales y los cálculos de M0/M1/M2 se conservan.

La ejecución de esta sección lee las referencias publicadas y ajusta únicamente M3. Guarda cuatro tablas con sus JSON y registra una configuración adicional por regla en el historial compartido. Republicar M0/M1 o M2 retira estas salidas para evitar mezclar resultados de intentos distintos. Si falla el guardado, repite la celda completa.

Incluso si mejora, M3 no queda aceptado por estos resultados solamente: siguen pendientes sus contrastes específicos y la simulación económica. No se retocarán la interacción, sus ventanas o el criterio de estabilidad según la comparación obtenida. Antes del commit, limpia las salidas y guarda.

In [ ]:
guardados_interaccion = []
for nombre, tabla in tablas_interaccion.items():
    salida = save_result(ROOT, nombre, tabla, informe_interaccion)
    guardados_interaccion.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados_interaccion))
print('Historial:', informe_interaccion['trial_log'])
print('Validación de M3 pendiente:', informe_interaccion['skill_gate_status'])

## 13. Retirar una variable cada vez: qué aporta dentro de M2

M2 reúne volatilidad, retorno de 24 meses y drawdown junto a las dos características de las señales. Ahora quitamos una de esas tres variables de régimen cada vez y **volvemos a entrenar** la logística con las cuatro columnas restantes. Conservamos años, etiquetas, ejemplos, purga y parámetros.

Es una comparación condicionada al resto de entradas. Una variable puede parecer prescindible porque otra recoge información similar. Su resultado tampoco demuestra causalidad ni una importancia válida para cualquier modelo. Este procedimiento se suele llamar *ablación* o retirada individual. Se diferencia de Clustered MDA: allí se desordena un grupo y se utiliza el modelo ya ajustado, aquí se elimina una columna y se ajusta otra vez.

Las tres variantes estaban previstas en el protocolo. No añadimos retiradas de combinaciones, otras ventanas o la interacción de M3. Se calcula un modelo por año, regla y retirada: 240 ajustes adicionales. M2 se lee como referencia, esta sección no lo reentrena ni cambia sus tablas.

In [ ]:
from tfm.walk_forward_ablation import REMOVAL_MODELS, run_walk_forward_ablation

nombres_retiradas = {'z_price_volatility_12m': 'Volatilidad',
    'z_market_return_24m': 'Retorno de 24 meses', 'z_drawdown_12m': 'Drawdown'}
display(pd.DataFrame([{'Variable retirada': nombres_retiradas[c], 'Modelo': m}
                      for c, m in REMOVAL_MODELS.items()]))

def progreso_retiradas(completadas, total):
    if completadas == 1 or completadas % 10 == 0:
        print(f'Bloques de variante y año evaluados: {completadas}/{total}', flush=True)

tablas_retiradas, informe_retiradas = run_walk_forward_ablation(ROOT, progress=progreso_retiradas)
comparacion_retiradas = tablas_retiradas['walk_forward_ablation_comparison']
print('Estado del recorrido:', informe_retiradas['experiment_status'])

## 14. Leer la contribución de conservar la variable

Usamos una orientación distinta a la de «modelo nuevo menos referencia», para describir la aportación de la variable retirada:

- **Contribución de AUC = AUC de M2 completo − AUC sin la variable.**
- **Contribución de log-loss = log-loss sin la variable − log-loss de M2 completo.**

En ambos casos una cifra positiva favorece conservarla. Por ejemplo, si M2 tiene AUC 0,60 y al quitar la variable baja a 0,58, la contribución es +0,02. Si al quitarla sube a 0,62, es −0,02. Un empate en AUC puede acompañarse de probabilidades y log-loss diferentes.

Cada retirada usa los mismos meses. Los años sin AUC estimable siguen presentes. Los recuentos muestran contribuciones positivas, negativas, empates y años no estimables, una mayoría descriptiva tampoco acredita aceptación del régimen.
El AUC reunido también compara ejemplos de años distintos y no es una media de las AUC anuales. Por eso la contribución global puede ser negativa aunque haya más años con contribución positiva que negativa.


In [ ]:
from matplotlib.ticker import MaxNLocator, FormatStrFormatter

principal_retiradas = comparacion_retiradas.loc[comparacion_retiradas.aggregation.eq('pooled')].copy()
principal_retiradas['Variable'] = principal_retiradas.removed_feature.map(nombres_retiradas)
display(principal_retiradas[['strategy', 'Variable', 'examples', 'auc_m2', 'auc_without',
    'auc_contribution', 'log_loss_m2', 'log_loss_without', 'log_loss_contribution', 'comparison_status']])
display(pd.DataFrame(informe_retiradas['contributions']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for ax, regla in zip(axes, ('momentum', 'reversal')):
    datos = principal_retiradas.loc[principal_retiradas.strategy.eq(regla)]
    contribucion = datos.auc_contribution.to_numpy(dtype=float)
    ax.barh(datos.Variable, contribucion, color=np.where(contribucion > 0, '#43876b', '#b65b49'))
    ax.axvline(0, color='black', linewidth=.8)
    ax.set(title=regla, xlabel='AUC de M2 − AUC sin la variable')
    ax.grid(axis='x', alpha=.2)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
fig.suptitle('Contribución condicionada a las demás entradas de M2')
fig.tight_layout()
plt.show()

## 15. Un diagnóstico, sin elegir automáticamente las variables

El análisis permite describir qué cambia al retirar cada columna. No convierte la mejor retirada observada en un modelo validado de forma independiente sobre estos mismos años. Si más adelante elegimos un subconjunto o una configuración por sus resultados, esa elección deberá realizarse dentro del entrenamiento y evaluarse por separado.

Se registran las tres variantes, incluidos resultados desfavorables. Los mismos meses aparecen una vez por variante: esas repeticiones no aumentan el número de observaciones independientes. El historial registra cada retirada antes de ajustarla, si falla una, se detiene el recorrido y no se publica una familia parcial. No se modifican M0/M1/M2/M3.

La siguiente celda guarda cuatro tablas `walk_forward_ablation_*`, con la columna retirada y el modelo identificados en cada fila. Republicar M0/M1 o M2 retira estas salidas, M3 no es una entrada y republicarlo las conserva. Si falla el guardado, repite la celda completa. La robustez CPCV equivalente se calcula a continuación. Siguen pendientes las sombras específicas, interpretación y evaluación económica.

In [ ]:
guardados_retiradas = []
for nombre, tabla in tablas_retiradas.items():
    salida = save_result(ROOT, nombre, tabla, informe_retiradas)
    guardados_retiradas.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados_retiradas))
print('Historial:', informe_retiradas['trial_log'])
print('Alcance:', informe_retiradas['skill_gate_status'])

## 16. Repetir las comparaciones con CPCV

Hasta aquí hemos probado años futuros usando solo entrenamiento anterior. Ahora comprobamos los mismos seis candidatos con las quince particiones del notebook 10: M1, M2, M3 y las tres retiradas de M2. Usamos las mismas columnas y parámetros, no elegimos los candidatos que hayan salido mejor en walk-forward.

Cada partición reserva dos de los seis bloques y entrena con los otros cuatro, respetando purga de un mes y embargo de seis. Puede aprender de bloques posteriores a su prueba, por eso es una comprobación de robustez sobre el desarrollo, no una reproducción de decisiones en tiempo real ni evidencia independiente.

La interacción de M3 y las otras transformaciones se calculan con el pasado de cada fecha antes de distribuir los bloques. No se normalizan los meses iniciales utilizando estadísticas de los bloques futuros de entrenamiento. La muestra común y sus limitaciones se mantienen.

**Entradas de esta sección:** 07/08 y las dos tablas del 10, preparadas a partir de 01/02. No necesita resultados de 11–13 ni de las secciones walk-forward. Si faltan las particiones, ejecuta el 10 antes de continuar. Los cálculos y la interpretación están en [el protocolo CPCV](../docs/cpcv_regimen.md).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display
from tfm.data import project_root, save_result
from tfm.cpcv_regime import CANDIDATE_COLUMNS, run_cpcv_regime

ROOT = project_root()
display(pd.DataFrame([{'Modelo': m, 'Predictores': ', '.join(c)} for m, c in CANDIDATE_COLUMNS.items()]))
def progreso_cpcv(completadas, total):
    if completadas == 1 or completadas % 15 == 0:
        print(f'Combinaciones de candidato y partición: {completadas}/{total}', flush=True)

tablas_cpcv, informe_cpcv = run_cpcv_regime(ROOT, progress=progreso_cpcv)
metricas_cpcv = tablas_cpcv['cpcv_regime_metrics']
comparacion_cpcv = tablas_cpcv['cpcv_regime_comparison']
print('Estado del recorrido:', informe_cpcv['experiment_status'])
display(pd.DataFrame(informe_cpcv['diagnostics']))

## 17. Cinco predicciones de cada ejemplo, sin multiplicar la muestra

Cada mes utilizable aparece en cinco particiones de prueba. Conservamos esas cinco probabilidades y, para el resumen de cada candidato, calculamos su media por regla y mes, igual que en el notebook 11. M0 se calcula con la frecuencia positiva del entrenamiento de cada partición y recibe el mismo tratamiento. Su referencia es idéntica para los seis candidatos y se guarda una vez en las métricas. La frecuencia puede variar entre particiones y su media mensual entre fechas, por eso su AUC reunida no tiene por qué ser 0,5.

Las cinco apariciones no son cinco observaciones independientes. Su media tampoco es una trayectoria CPCV de rentabilidades ni una cartera. Si falta una de las cinco probabilidades, su media queda ausente y no se calcula una métrica ignorando ese mes. Conservamos las particiones sin ejemplos o con una clase, indicando cuándo AUC no puede calcularse.

Mostramos dos resúmenes, fijados antes del cálculo: `monthly_mean` usa todo el desarrollo utilizable, `common_period` restringe esas mismas medias a 1950–1989, los meses de prueba del walk-forward. Este segundo resumen no reentrena ni cambia los bloques. Alinear meses no iguala la información de entrenamiento ni convierte ambos procedimientos en pruebas independientes. Por eso no se comparan sus AUC como si solo cambiara el modelo.

In [ ]:
from matplotlib.ticker import MaxNLocator

display(metricas_cpcv.loc[metricas_cpcv.aggregation.isin(['monthly_mean', 'common_period']),
    ['aggregation', 'strategy', 'model', 'examples', 'log_loss', 'auc', 'status']])
display(comparacion_cpcv.loc[comparacion_cpcv.aggregation.isin(['monthly_mean', 'common_period']),
    ['aggregation', 'reference_model', 'candidate_model', 'removed_feature', 'strategy', 'examples',
     'auc_reference', 'auc_candidate', 'auc_improvement', 'log_loss_improvement', 'comparison_status']])
display(pd.DataFrame(informe_cpcv['stability']))

principal_cpcv = comparacion_cpcv.loc[comparacion_cpcv.aggregation.eq('monthly_mean') &
    comparacion_cpcv.removed_feature.isna()].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for ax, regla in zip(axes, ('momentum', 'reversal')):
    datos = principal_cpcv.loc[principal_cpcv.strategy.eq(regla)]
    etiquetas = datos.candidate_model + ' frente a ' + datos.reference_model
    cambio = datos.auc_improvement.to_numpy(dtype=float)
    ax.barh(etiquetas, cambio, color=np.where(cambio > 0, '#43876b', '#b65b49'))
    ax.axvline(0, color='black', linewidth=.8)
    ax.set(title=regla, xlabel='AUC del candidato − AUC de referencia')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.grid(axis='x', alpha=.2)
fig.suptitle('Comparaciones CPCV sobre todo el desarrollo utilizable')
fig.tight_layout()
plt.show()

## 18. Qué añade esta comprobación y qué sigue pendiente

La tabla separa la aportación del bloque (M2 frente a M1), la interacción (M3 frente a M2 y M1) y la contribución de mantener cada variable (M2 frente al modelo que la retira). Una diferencia positiva favorece siempre al candidato indicado. En las retiradas ese candidato es M2 completo, por lo que una cifra positiva favorece conservar la variable.

La estabilidad se describe sobre las quince particiones: mayoría estricta significa al menos ocho con mejora. Las particiones comparten meses y episodios, así que no son quince experimentos independientes. El resumen común de 1950–1989 ayuda a interpretar diferencias de muestra, pero no sustituye el resultado CPCV completo ni la evaluación cronológica.

Se mantienen las listas fijadas y no se cambia una decisión por escoger ahora la validación más favorable. CPCV utiliza las mismas configuraciones ya registradas: su historial identifica la nueva validación, sin contabilizar otras seis listas o hiperparámetros nuevos. No se ha calculado DSR/PBO a partir del número de particiones. Tampoco una mejora conjunta de estas métricas acredita por sí sola rentabilidad o aceptación del régimen: quedan las sombras específicas, la estabilidad exigida en ambos esquemas y la evaluación económica.

Esta sección guarda cinco tablas `cpcv_regime_*` e historial separado, sin modificar los resultados originales del 11–13 o del walk-forward. Recalcular características, etiquetas o las particiones del 10 retira estas salidas. Republicar los modelos anteriores las conserva, porque no son sus entradas. Si falla el guardado, repite la celda completa. El periodo desde 1990 sigue reservado. Antes del commit, limpia las salidas y guarda.

In [ ]:
guardados_cpcv = []
for nombre, tabla in tablas_cpcv.items():
    salida = save_result(ROOT, nombre, tabla, informe_cpcv)
    guardados_cpcv.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados_cpcv))
print('Historial:', informe_cpcv['trial_log'])
print('Validación pendiente:', informe_cpcv['skill_gate_status'])

## 19. De probabilidades a decisiones de inversión

Ahora usamos las predicciones **walk-forward**, una por operación activa, para simular cuentas continuas. No usamos las medias CPCV ni capitalizamos los retornos aislados del etiquetado. Esta sección puede ejecutarse en un kernel nuevo con 01/02/07/08 y los resultados M0/M1, M2 y M3 ya preparados, no requiere repetir ajustes, sombras, CPCV o las retiradas individuales.

Fijamos el criterio antes de calcular estas curvas: aceptar la señal original si su probabilidad es **mayor que 0,5**, con 0,5 o menos, mantener efectivo. Por ejemplo, una reversión corta con probabilidad 0,62 sigue siendo corta, con 0,43 se rechaza. El modelo no cambia la dirección ni el tamaño de una señal aceptada. Una señal original cero sigue siendo efectivo y no necesita predicción. Si falta una probabilidad para una señal activa, el cálculo se detiene.

El 50 % es una convención de clasificación, no un umbral óptimo demostrado: acertar muchas operaciones pequeñas no garantiza compensar unas pocas pérdidas grandes. Tampoco asegura que las probabilidades estén calibradas. No buscaremos ahora otro umbral que mejore estas curvas. El diseño completo está en [el protocolo económico](../docs/evaluacion_economica.md).


In [ ]:
from tfm.data import project_root, save_result
from tfm.economics import run_economics
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = project_root()
tablas_economicas, informe_economico = run_economics(ROOT)
decisiones_economicas = tablas_economicas['walk_forward_economic_decisions']
cuentas_economicas = tablas_economicas['walk_forward_economic_simulations']
metricas_economicas = tablas_economicas['walk_forward_economic_metrics']
comparacion_economica = tablas_economicas['walk_forward_economic_comparison']
display(pd.DataFrame([informe_economico['parameters']]))
display(pd.DataFrame([informe_economico['decision_policy']]))
display(metricas_economicas.loc[metricas_economicas.active_signals.notna(),
    ['label', 'active_signals', 'accepted_signals', 'rejected_signals']])


## 20. Las cuentas empiezan y terminan en las mismas fechas

Todas parten de efectivo con capital 100 en enero de 1950 y terminan con liquidación en diciembre de 1989. Conservamos los 480 meses, incluidos los rechazados y los meses sin señal. El reentrenamiento anual no reinicia el capital ni cierra automáticamente una posición en diciembre.

Reutilizamos el mismo motor contable: dividendos aproximados, remuneración RF del efectivo, negociación sobre el nominal realmente cambiado y préstamo de los cortos. Rechazar una señal puede requerir cerrar una posición anterior y pagar su coste. Aceptar dos señales consecutivas no fuerza venta y recompra mensual. Las etiquetas evaluaban operaciones aisladas, estas curvas evalúan posiciones continuas y por eso sus retornos no se pueden intercambiar.

Se recalculan las referencias con el mismo inicio y final: comprar y mantener, efectivo, reglas sin filtrar, veto bajista y ajuste por volatilidad. Este último mantiene ventana de 12 meses, objetivo anual del 12 % y exposición máxima uno. Ninguna comparación reutiliza una curva que ya hubiera acumulado capital desde 1927.

La tabla conserva las dieciséis cuentas, incluidos los filtros M0. El gráfico muestra M1–M3 y las referencias sin filtro, por volatilidad y en efectivo, las demás cuentas siguen disponibles en las tablas. Cada panel usa su propia escala vertical para apreciar los cambios dentro de su regla, no para comparar alturas entre paneles. Las curvas usan precios medios mensuales de Shiller: son aproximaciones históricas, no operaciones ejecutables a precios de cierre.


In [ ]:
resumen_economico = metricas_economicas[['label', 'capital_final', 'annualized_return',
    'annualized_volatility', 'sharpe', 'max_drawdown', 'annualized_turnover',
    'transaction_cost_total', 'borrow_cost_total']].copy()
# Retorno, volatilidad y drawdown se muestran como porcentajes, los costes, en unidades de capital.
for columna in ('annualized_return', 'annualized_volatility', 'max_drawdown'):
    resumen_economico[columna] *= 100
display(resumen_economico)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=False)
colores = {'': '#555555', '_volatility': '#bb9135', '_m1': '#3069a3',
           '_m2': '#43876b', '_m3': '#a15f90'}
for ax, regla, titulo in zip(axes, ('momentum', 'reversal'), ('Momentum', 'Reversión')):
    curvas = [(regla + sufijo, color) for sufijo, color in colores.items()] + [('cash', '#777777')]
    for cuenta, color in curvas:
        datos = cuentas_economicas.loc[cuentas_economicas.strategy.eq(cuenta)]
        etiqueta = informe_economico['accounts'][cuenta]
        fechas = pd.PeriodIndex(datos.period, freq='M').to_timestamp(how='end')
        # El punto inicial permite ver el coste de apertura sin perderlo en el primer mes.
        inicio = pd.Timestamp(informe_economico['test_interval'][0] + '-01')
        ax.plot([inicio, *fechas], [informe_economico['parameters']['initial_capital'], *datos.capital_end],
                label=etiqueta, color=color, linewidth=1.3,
                linestyle=':' if cuenta == 'cash' else '--' if cuenta in (regla, regla + '_volatility') else '-')
    ax.set(title=titulo, xlabel='Año', ylabel='Capital de la cuenta (inicio = 100)')
    ax.grid(alpha=.2)
    ax.legend(fontsize=8)
fig.suptitle('Cuentas continuas de desarrollo · 1950–1989 · después de costes')
fig.tight_layout()
plt.show()


## 21. Qué compara el Sharpe y qué no demuestra

El Sharpe resume el exceso mensual de la cuenta sobre RF dividido por su variabilidad, anualizado con raíz de doce. Mayor es favorable, pero no demuestra por sí solo significación ni una mejora estable. Una cuenta siempre en efectivo tiene exceso cero y Sharpe no estimable, no lo reemplazamos por infinito ni lo tratamos como una victoria.

Las diferencias siguientes son Sharpe del candidato menos Sharpe de su referencia. Cada filtro se compara con su regla sin filtrar, el veto y la escala por volatilidad. Además contrastamos M2 y M3 con M1, y M3 con M2. Conservamos también diferencias desfavorables y comparaciones no estimables. No cambiamos el umbral, la muestra o los modelos al ver la tabla.

M2 y M3 no alcanzaron la mayoría anual de mejoras predictivas exigida. Una curva o un Sharpe favorable no elimina ese incumplimiento y estas tablas no aceptan el régimen. Las sombras específicas que correspondan, los demás candidatos y los controles de sobreajuste siguen siendo trabajos separados. No calculamos DSR/PBO a partir de probabilidades ni generamos SHAP con estas curvas. El periodo desde 1990 continúa reservado.


In [ ]:
display(comparacion_economica[['rule', 'model', 'reference', 'sharpe_candidate',
    'sharpe_reference', 'sharpe_improvement', 'comparison_status']])


## 22. Guardar el diagnóstico económico

Guardamos las decisiones, las cuentas mensuales, sus métricas y las comparaciones. El historial identifica el umbral fijo y los resultados predictivos utilizados, no se ajustan modelos nuevos. Repetir el cálculo con las mismas entradas no crea otra configuración.

Las cuatro tablas económicas se retiran si regeneramos sus datos o republicamos M0/M1, M2 o M3. Publicar CPCV, sombras o las retiradas individuales las conserva porque no son sus entradas. Esta sección no modifica las cuentas ni los experimentos anteriores. Si falla el guardado, repite la celda completa. Antes del commit, limpia las salidas y guarda el notebook.


In [ ]:
guardados_economicos = []
for nombre, tabla in tablas_economicas.items():
    salida = save_result(ROOT, nombre, tabla, informe_economico)
    guardados_economicos.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados_economicos))
print('Historial:', informe_economico['trial_log'])
print('Alcance:', informe_economico['regime_acceptance'])


## 23. Segundo candidato: un random forest pequeño

La logística transforma una suma de variables en una probabilidad. Un árbol plantea preguntas sucesivas: por ejemplo, si la volatilidad es alta y después si el retorno anterior es negativo. Un bosque combina muchos árboles y promedia sus probabilidades. Esto permite representar relaciones no lineales, pero también aprender casualidades del pasado si le damos demasiada libertad.

Fijamos antes del ajuste **200 árboles, profundidad máxima 3, mínimo 20 observaciones por hoja y semilla 42**. Cada árbol puede tener como máximo ocho hojas. Todos los candidatos usan estos parámetros, bootstrap del entrenamiento y todas sus columnas como opciones de división. No buscamos otros parámetros ni equilibramos clases artificialmente. Una hoja puede dar una probabilidad poco fiable, el promedio no garantiza calibración.

Comparamos **RF_M1**, con las dos variables de señal, **RF_M2**, que añade las tres variables de régimen, y **RF_M3**, que incorpora además el producto mercado bajista × volatilidad. RF_M2 ya puede aprender interacciones. RF_M3 comprueba si añadir ese producto explícito facilita la representación, no si aparecen interacciones por primera vez.

Conservamos las transformaciones y la muestra común para comparar familias sin cambiar también la preparación de datos. Un árbol no necesita z-score, pero la transformación común y la winsorización siguen formando parte del experimento fijado. No se usa el cribado para seleccionar columnas ni se toman importancias Gini como sustituto de Clustered MDA.

**Entradas:** 01/02/07/08 y resultados M0/M1/M2 logísticos. Esta sección puede ejecutarse en un kernel nuevo, no requiere M3 logístico, CPCV, sombras o cuentas económicas. No vuelve a ajustar la logística. El diseño y las limitaciones están en [el protocolo del bosque](../docs/random_forest.md).


In [ ]:
from tfm.data import project_root, save_result
from tfm.forest import run_forest, FOREST_PARAMETERS, CANDIDATE_COLUMNS
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display

ROOT = project_root()
display(pd.DataFrame([FOREST_PARAMETERS]))
display(pd.DataFrame([{'Modelo': modelo, 'Columnas': ', '.join(columnas)}
    for modelo, columnas in CANDIDATE_COLUMNS.items()]))

def progreso_bosque(completados, total):
    if completados % 10 == 0 or completados == total:
        print(f'Combinaciones de candidato y año: {completados}/{total}')

tablas_bosque, informe_bosque = run_forest(ROOT, progress=progreso_bosque)
metricas_bosque = tablas_bosque['forest_walk_forward_metrics']
comparacion_bosque = tablas_bosque['forest_walk_forward_comparison']
display(metricas_bosque.loc[metricas_bosque.aggregation.eq('pooled'),
    ['strategy', 'model', 'examples', 'positive', 'negative', 'auc', 'log_loss', 'status']])


## 24. Qué añade el bosque y qué aporta el régimen dentro del bosque

Repetimos los cuarenta años de prueba de 1950–1989, con entrenamiento expansivo hasta noviembre del año anterior y diciembre purgado. La semilla y los parámetros son iguales para las tres listas. Los resultados de cada año usan únicamente sus ejemplos reservados, las métricas reunidas usan una predicción por operación, sin medias CPCV ni promedios de AUC anuales.

Separamos dos preguntas. RF_M1 frente a L2_M1 y RF_M2 frente a L2_M2 comparan **familias de modelos con las mismas columnas**. RF_M2 frente a RF_M1, RF_M3 frente a RF_M1 y RF_M3 frente a RF_M2 miden la aportación de añadir entradas dentro del bosque. La logística mantiene sus resultados originales.

En las tablas y el gráfico, una diferencia positiva favorece al candidato: AUC del candidato menos referencia, o log-loss de referencia menos candidato. Un año vacío o con una sola clase sigue presente, si no puede calcularse AUC, no cuenta como mejora. Se necesitan al menos veintiún años favorables de los cuarenta previstos para la mayoría establecida, sin cambiar el denominador.

Una mejora frente a la logística no acepta automáticamente el régimen. Tampoco un buen resultado estadístico demuestra una mejora de inversión: CPCV equivalente, sombras específicas y cuentas económicas del bosque siguen pendientes. Las sombras del modelo logístico anterior no validan este nuevo candidato. No elegimos ahora el mejor bosque o eliminamos variables al mirar estos resultados.


In [ ]:
display(comparacion_bosque.loc[comparacion_bosque.aggregation.eq('pooled'),
    ['reference_model', 'candidate_model', 'strategy', 'auc_reference', 'auc_candidate',
     'auc_improvement', 'log_loss_improvement', 'comparison_status']])
display(pd.DataFrame(informe_bosque['stability']))
display(tablas_bosque['forest_walk_forward_diagnostics'].groupby('candidate', sort=False).agg(
    ajustes=('status', 'size'), profundidad_maxima=('max_observed_depth', 'max'),
    hojas_medias=('mean_leaves', 'mean'), estado=('status', lambda x: ', '.join(sorted(set(x))))).reset_index())

principal_bosque = comparacion_bosque.loc[comparacion_bosque.aggregation.eq('pooled')]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
for ax, regla, titulo in zip(axes, ('momentum', 'reversal'), ('Momentum', 'Reversión')):
    datos = principal_bosque.loc[principal_bosque.strategy.eq(regla)]
    etiquetas = datos.candidate_model + ' frente a ' + datos.reference_model
    cambios = datos.auc_improvement.to_numpy(dtype=float)
    ax.barh(etiquetas, cambios, color=['#43876b' if x > 0 else '#b65b49' for x in cambios])
    ax.axvline(0, color='black', linewidth=.8)
    ax.set(title=titulo, xlabel='AUC del candidato − AUC de referencia')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.grid(axis='x', alpha=.2)
fig.suptitle('Random forest · comparaciones cronológicas de desarrollo')
fig.tight_layout()
plt.show()


## 25. Guardar el segundo candidato y conservar sus límites

Se guardan las predicciones, las métricas, el diagnóstico de los ajustes y las cinco comparaciones. Cada lista constituye una nueva configuración por regla, los 200 árboles y cuarenta años no son configuraciones independientes. El historial registra cada intento antes de ajustar y conserva resultados o fallos. Si una lista falla, el recorrido se detiene sin devolver una familia parcial para publicar.

Estas tablas dependen de las características, etiquetas y referencias M0/M1/M2 logísticas. Regenerar sus entradas o republicar esas referencias retira las tablas del bosque. Publicar M3 logístico, CPCV, sombras, retiradas o cuentas económicas las conserva porque no son entradas de este cálculo. Publicar el bosque mantiene intactos los resultados anteriores.

Quedan pendientes las comprobaciones propias del bosque y el siguiente candidato, gradient boosting pequeño. EBM seguirá condicionado al estudio de relaciones no lineales. No se han generado SHAP, selección automática o controles DSR/PBO, y el periodo desde 1990 sigue reservado. Si falla el guardado, repite esta celda completa. Antes del commit, limpia salidas y guarda.


In [ ]:
guardados_bosque = []
for nombre, tabla in tablas_bosque.items():
    salida = save_result(ROOT, nombre, tabla, informe_bosque)
    guardados_bosque.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados_bosque))
print('Historial:', informe_bosque['trial_log'])
print('Estado:', informe_bosque['experiment_status'])
print('Alcance:', informe_bosque['skill_gate_status'])


## 26. Diagnóstico anual de los modelos ya calculados

Este bloque responde a la revisión con el tutor: **ver cada año de prueba, cuántas operaciones contiene y qué entorno lo acompaña**. Puede ejecutarse desde un kernel nuevo con las salidas de logística M0–M3 y random forest del 14 ya guardadas. Basta con ejecutar las secciones 26–29 para consultar este diagnóstico.

Leemos las probabilidades guardadas, comprobamos que comparten señales y referencias, y reconstruimos sus métricas. Las tablas originales permanecen como fuentes de resultados. Este bloque presenta información descriptiva: conserva las configuraciones y el periodo de evaluación fijados.

Distinguimos dos conjuntos: **los doce meses del mercado** y **los meses con señal de cada regla**. Sus medias de contexto pueden ser diferentes. Una AUC anual necesita operaciones de las dos clases, favorables y desfavorables, mantenemos los cuarenta años aunque alguna AUC quede ausente.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

from tfm.data import project_root
from tfm.annual_diagnostics import MODELS, load_annual_diagnostics

ROOT = project_root()
diagnostico_anual = load_annual_diagnostics(ROOT)
metricas_anuales = diagnostico_anual['annual']
operaciones_anuales = diagnostico_anual['operations']
contexto_anual = diagnostico_anual['market']
ANIOS = list(range(1950, 1990))
NOMBRES_REGLAS = {'momentum': 'Momentum', 'reversal': 'Reversión'}

print('Diagnóstico: 40 años, 2 reglas y 7 métodos (M0, tres logísticas y tres bosques).')
print('Los modelos L2_M1/M2/M3 y RF_M1/M2/M3 utilizan las listas del mismo nombre.')
display(operaciones_anuales.groupby('strategy').agg(
    operaciones=('examples', 'sum'), favorables=('positive', 'sum'),
    desfavorables=('negative', 'sum'),
    anios_con_auc_calculable=('auc_status', lambda s: s.eq('available').sum())))

## 27. Ver la AUC de cada año

Cada fila es un año y cada columna un método. **L2** identifica la logística y **RF**, el bosque. La escala de color es siempre de 0 a 1, con centro en 0,5. La cifra de una celda muestra su AUC, una raya sobre fondo gris identifica un año cuya AUC queda ausente. La sección siguiente permite ver si faltan operaciones o si todas tienen la misma etiqueta.

Para comparar familias, miramos la misma lista: L2_M1 frente a RF_M1, por ejemplo. Para estudiar el entorno, comparamos M2 o M3 con M1 dentro de la misma familia. La AUC reunida combina pares de todos los años y se calcula por separado: aquí observamos los resultados anuales, con muestras pequeñas de hasta doce operaciones por regla.

In [ ]:
cmap_auc = plt.get_cmap('RdYlGn').copy()
cmap_auc.set_bad('#e3e3e3')
fig, axes = plt.subplots(1, 2, figsize=(13, 14), sharey=True)
for ax, (regla, titulo) in zip(axes, NOMBRES_REGLAS.items()):
    matriz = metricas_anuales.loc[metricas_anuales.strategy.eq(regla)].pivot(
        index='test_year', columns='model', values='auc').reindex(index=ANIOS, columns=list(MODELS))
    valores = matriz.to_numpy(dtype=float, na_value=np.nan)
    imagen = ax.imshow(np.ma.masked_invalid(valores), aspect='auto', cmap=cmap_auc,
        norm=TwoSlopeNorm(vmin=0, vcenter=.5, vmax=1), interpolation='nearest')
    ax.set_xticks(range(len(MODELS)), list(MODELS), rotation=45, ha='right')
    ax.set_yticks(range(len(ANIOS)), ANIOS)
    ax.tick_params(axis='y', labelsize=8)
    ax.set(title=titulo, xlabel='Método', ylabel='Año de prueba')
    for fila in range(len(ANIOS)):
        for columna in range(len(MODELS)):
            valor = valores[fila, columna]
            etiqueta = '—' if np.isnan(valor) else f'{valor:.2f}'
            color = 'white' if np.isfinite(valor) and (valor < .15 or valor > .85) else '#252525'
            ax.text(columna, fila, etiqueta, ha='center', va='center', fontsize=7, color=color)
fig.subplots_adjust(left=.07, right=.86, bottom=.09, top=.93, wspace=.22)
eje_color = fig.add_axes([.9, .2, .018, .6])
fig.colorbar(imagen, cax=eje_color, label='AUC anual · centro 0,5')
fig.suptitle('Logística y random forest · AUC de cada año de prueba', y=.985)
plt.show()

## 28. Cuántas operaciones sostienen cada resultado y cómo era el entorno

El primer gráfico cuenta las operaciones favorables y desfavorables. Las barras con una sola clase explican las rayas grises de la AUC. Con pocas operaciones, uno o dos cambios de orden pueden mover mucho la AUC: el tamaño de muestra acompaña siempre su interpretación.

El segundo gráfico resume **los doce meses de cada año objetivo**, usando las características que correspondían a sus decisiones:

- Meses bajistas: número de meses cuyo retorno previo de 24 meses era negativo.
- Media de volatilidad: promedio de los doce indicadores de volatilidad de 12 meses, anualizados.
- Distancia media al máximo: promedio de las doce distancias al máximo reciente de precios de 12 meses.

Estas medias describen el año una vez completo. El modelo recibía sus características mensuales preparadas con el pasado. La distancia al máximo pertenece al índice, las caídas de la cuenta de inversión se estudian en la evaluación económica.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, sharey=True)
for ax, (regla, titulo) in zip(axes, NOMBRES_REGLAS.items()):
    datos = operaciones_anuales.loc[operaciones_anuales.strategy.eq(regla)].set_index('test_year').reindex(ANIOS)
    positivas = datos.positive.to_numpy(dtype=int)
    negativas = datos.negative.to_numpy(dtype=int)
    ax.bar(ANIOS, positivas, color='#43876b', label='Favorables: etiqueta 1')
    ax.bar(ANIOS, negativas, bottom=positivas, color='#c98168', label='Desfavorables: etiqueta 0')
    ax.set(title=titulo, ylabel='Operaciones', ylim=(0, 12.5))
    ax.set_yticks([0, 3, 6, 9, 12])
    ax.grid(axis='y', alpha=.2)
axes[0].legend(loc='lower left', bbox_to_anchor=(0, 1.14), ncols=2)
axes[-1].set_xlabel('Año de prueba')
axes[-1].set_xlim(1949.4, 1989.6)
axes[-1].set_xticks([*range(1950, 1990, 5), 1989])
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
axes[0].bar(contexto_anual.test_year, contexto_anual.bear_months, color='#92715b')
axes[0].set(ylabel='Meses bajistas', ylim=(0, 12.5))
axes[0].set_yticks([0, 3, 6, 9, 12])
axes[1].plot(contexto_anual.test_year, 100*contexto_anual.mean_volatility_12m,
    marker='o', markersize=3, color='#497b9b')
axes[1].set_ylabel('Volatilidad media\nde 12 meses (%)')
axes[2].plot(contexto_anual.test_year, 100*contexto_anual.mean_drawdown_12m,
    marker='o', markersize=3, color='#846494')
axes[2].axhline(0, color='black', linewidth=.7)
axes[2].set(ylabel='Distancia media al máximo\nde 12 meses (%)', xlabel='Año objetivo')
for ax in axes:
    ax.grid(axis='y', alpha=.2)
    ax.set_xticks([*range(1950, 1990, 5), 1989])
    ax.set_xlim(1949.4, 1989.6)
fig.suptitle('Contexto del año · doce meses del calendario, con información mensual anterior')
fig.tight_layout()
plt.show()

## 29. Leer un año con sus cifras y separar descripción de decisión

Cambia `ANIO_A_REVISAR` para consultar un año. La primera tabla muestra el mercado completo y la segunda el contexto de las operaciones de cada regla. Un año puede contener varios entornos y momentum puede operar solo en parte de ellos.

La última tabla reúne las AUC y log-loss de los siete métodos. El estado explica por qué una AUC queda ausente. Las cifras se presentan redondeadas, se mantienen las métricas completas en las tablas leídas.

Este diagnóstico permite formular preguntas sobre por qué varía el comportamiento. Los cuarenta años, los criterios de estabilidad y las listas se conservan. Las asociaciones que observemos aquí son exploratorias y deberán declararse al planear nuevos candidatos. Los controles propios del bosque y los pasos posteriores siguen pendientes.

In [ ]:
ANIO_A_REVISAR = 1950
if ANIO_A_REVISAR not in ANIOS:
    raise ValueError('Elige un año entre 1950 y 1989.')

mercado_vista = contexto_anual.loc[contexto_anual.test_year.eq(ANIO_A_REVISAR)].copy()
mercado_vista[['mean_volatility_12m', 'mean_drawdown_12m']] *= 100
print(f'{ANIO_A_REVISAR} · contexto de los doce meses del mercado')
display(mercado_vista.rename(columns={
    'test_year': 'Año', 'calendar_months': 'Meses de mercado', 'bear_months': 'Meses bajistas',
    'mean_volatility_12m': 'Volatilidad media (%)', 'mean_drawdown_12m': 'Distancia media al máximo (%)'}).round(3))

operaciones_vista = operaciones_anuales.loc[operaciones_anuales.test_year.eq(ANIO_A_REVISAR)].copy()
operaciones_vista[['active_mean_volatility_12m', 'active_mean_drawdown_12m']] *= 100
print('Contexto limitado a los meses con operación de cada regla')
display(operaciones_vista[['strategy', 'examples', 'positive', 'negative', 'active_bear_months',
    'active_mean_volatility_12m', 'active_mean_drawdown_12m']].rename(columns={
    'strategy': 'Regla', 'examples': 'Operaciones', 'positive': 'Favorables', 'negative': 'Desfavorables',
    'active_bear_months': 'Operaciones en entorno bajista',
    'active_mean_volatility_12m': 'Volatilidad media en señales (%)',
    'active_mean_drawdown_12m': 'Distancia media en señales (%)'}).round(3))

estados_auc = {'available': 'Calculable', 'single_class': 'Una sola clase',
    'no_examples': 'Sin operaciones', 'unavailable_predictions': 'Faltan predicciones'}
for regla, titulo in NOMBRES_REGLAS.items():
    vista = metricas_anuales.loc[metricas_anuales.test_year.eq(ANIO_A_REVISAR)
        & metricas_anuales.strategy.eq(regla)].set_index('model').reindex(list(MODELS))
    vista = vista[['examples', 'positive', 'negative', 'auc', 'log_loss', 'auc_status']].copy()
    vista['auc_status'] = vista.auc_status.map(estados_auc)
    print(f'{titulo} · métricas de los mismos ejemplos')
    display(vista.rename(columns={'examples': 'Operaciones', 'positive': 'Favorables',
        'negative': 'Desfavorables', 'auc': 'AUC', 'log_loss': 'Log-loss',
        'auc_status': 'Estado AUC'}).round(4))

## 30. Qué características pueden estar contando lo mismo

Comprobamos la correlación de las diez entradas dentro del entrenamiento de cada año y regla. Dos variables pueden moverse en el mismo sentido o en sentidos opuestos. Si la correlación absoluta alcanza 0,80, pueden formar un grupo. Para agrupar tres o más, todas las parejas deben alcanzar ese corte. La tabla incluye las parejas que llegaron alguna vez a 0,70 y cuántas veces acabaron en el mismo grupo.

Ejecuta desde aquí si ya tienes guardada la referencia M0/M1 de este cuaderno. Se necesitan las tablas de 01, 02, 07 y 08. El cálculo completo de los cuatro escalones se prepara en esta celda.

[Protocolo de comparación](../docs/comparacion_bloques.md)

In [ ]:
from IPython.display import display
from tfm.data import project_root
from tfm.feature_blocks import run_feature_blocks

resultados_bloques = run_feature_blocks(project_root())
correlaciones = resultados_bloques['correlation_summary']
display(correlaciones.loc[correlaciones.years_near.add(correlaciones.years_strong).gt(0)]
    .rename(columns={'strategy': 'Regla', 'feature_a': 'Variable A', 'feature_b': 'Variable B',
        'median_spearman': 'Spearman mediano', 'years_available': 'Años calculables',
        'years_near': 'Años entre 0,70 y 0,80', 'years_strong': 'Años desde 0,80',
        'years_grouped': 'Años en el mismo grupo'}).round(3))

## 31. Cuánto añade cada bloque

Partimos de M1. Añadimos distancia a la media de 12 meses, después el nivel y el cambio del diferencial crediticio, a continuación amplitud y dispersión de las industrias y, al final, volatilidad, retorno de 24 meses y distancia al máximo. Cada fila compara un escalón con el inmediatamente anterior. Mantenemos los mismos años, señales y parámetros de la logística.

Una diferencia positiva favorece al nuevo escalón. Buscamos que suba el AUC reunido, baje la log-loss reunida y mejore el AUC en al menos 21 de los 40 años. Los años con empate o sin AUC tienen su propio recuento. Este resultado mide la aportación del bloque dadas las entradas ya presentes.

In [ ]:
resumen_bloques = resultados_bloques['summary']
nombres = {'M1': 'M1', 'Con_media': 'Con media', 'Con_credito': 'Con crédito',
           'Con_industrias': 'Con industrias', 'Con_regimen': 'Con régimen'}
columnas = {'strategy': 'Regla', 'reference_model': 'Referencia', 'candidate_model': 'Candidato',
    'auc_reference': 'AUC anterior', 'auc_candidate': 'AUC nuevo', 'auc_improvement': 'Mejora AUC',
    'log_loss_improvement': 'Mejora log-loss', 'auc_improved_years': 'Años que mejora',
    'auc_worsened_years': 'Años que empeora', 'auc_tied_years': 'Empates',
    'auc_unavailable_years': 'Años sin AUC', 'predictive_support': 'Cumple criterio predictivo'}
display(resumen_bloques[list(columnas)].replace(
    {'reference_model': nombres, 'candidate_model': nombres}).rename(columns=columnas).round(4))

## 32. Tercer candidato: gradient boosting

El modelo empieza con la frecuencia de éxitos del entrenamiento. Después añade árboles uno a uno. Cada árbol aprende un ajuste para reducir el error que queda y modifica un poco la probabilidad estimada. Limitamos la secuencia a 100 árboles pequeños de profundidad máxima 2, con al menos 20 ejemplos por hoja. La tasa de aprendizaje de 0,05 limita la aportación de cada árbol.

Ajustamos dos versiones por regla. **GB_M1** recibe retorno del último mes y momentum 12–1. **GB_M2** añade volatilidad, retorno de 24 meses y distancia al máximo. Cada año, las predicciones de boosting se calculan con los mismos ejemplos de entrenamiento y prueba que en la logística y el bosque.

Ejecuta desde esta sección con las salidas de 01/02/07/08, M0/M1/M2 y random forest ya guardadas. [Protocolo](../docs/gradient_boosting.md)

In [ ]:
import pandas as pd
from IPython.display import display
from tfm.boosting import BOOSTING_PARAMETERS, run_boosting
from tfm.data import project_root, save_result

ROOT = project_root()
display(pd.DataFrame([BOOSTING_PARAMETERS]))

def progreso_boosting(hechas, total):
    if hechas % 10 == 0 or hechas == total:
        print(f'Combinaciones de variante y año: {hechas}/{total}')

tablas_boosting, informe_boosting = run_boosting(ROOT, progress=progreso_boosting)
metricas_boosting = tablas_boosting['boosting_walk_forward_metrics']
metricas_globales = metricas_boosting.loc[metricas_boosting.aggregation.eq('pooled')]
display(metricas_globales[['strategy', 'model', 'examples', 'auc', 'log_loss']].round(5))

## 33. Qué cambia respecto a la logística y el bosque

Las filas GB frente a L2 o RF mantienen las mismas variables. GB_M2 frente a GB_M1 mide qué aporta añadir el bloque de régimen dentro de boosting. Los valores positivos de mejora favorecen al candidato. El criterio combina mejora de las dos métricas reunidas y mejora del AUC en al menos 21 de los 40 años. Los empates y años sin AUC se cuentan por separado.

In [ ]:
comparaciones_boosting = tablas_boosting['boosting_walk_forward_comparison']
resumen_boosting = comparaciones_boosting.loc[comparaciones_boosting.aggregation.eq('pooled')].merge(
    pd.DataFrame(informe_boosting['stability']),
    on=['reference_model', 'candidate_model', 'strategy'], validate='one_to_one')
nombres = {'strategy': 'Regla', 'reference_model': 'Referencia', 'candidate_model': 'Candidato',
    'auc_improvement': 'Mejora AUC', 'log_loss_improvement': 'Mejora log-loss',
    'auc_improved_years': 'Años que mejora', 'auc_worsened_years': 'Años que empeora',
    'auc_tied_years': 'Empates', 'auc_unavailable_years': 'Años sin AUC',
    'predictive_support': 'Cumple criterio predictivo'}
display(resumen_boosting[list(nombres)].rename(columns=nombres).round(5))

## 34. Guardar las predicciones y la comparación

Las probabilidades se guardan por regla y mes de prueba. Así se podrán usar para simular las operaciones después de costes. El historial cuenta una configuración por variante y regla.

In [ ]:
guardados_boosting = []
for nombre, tabla in tablas_boosting.items():
    guardado = save_result(ROOT, nombre, tabla, informe_boosting)
    guardados_boosting.append({'Tabla': nombre + '.parquet', 'Filas': guardado['rows']})
display(pd.DataFrame(guardados_boosting))
print('Historial:', informe_boosting['trial_log'])

## 35. Qué forma aprenden los modelos con las dos entradas de M1

Comparamos logística, random forest y boosting en momentum y reversión. Reproducimos los modelos disponibles al empezar 1959, 1969, 1979 y 1989 y comprobamos sus probabilidades con las guardadas. Los gráficos usan las señales activas de los últimos 120 meses de entrenamiento de cada modelo. Ejecuta desde aquí con las predicciones de las secciones 32–34 ya guardadas.

Para dibujar cada variable, tomamos las señales de un tramo y preguntamos cuánto cambia su puntuación al pasar de un extremo del tramo al otro, dejando la segunda entrada en su valor observado. Promediamos esos cambios y los sumamos desde el corte central. La logística tiene pendiente constante en todo el gráfico porque estamos mirando la puntuación anterior a la sigmoide. [Método y fechas](../docs/formas_m1.md)

In [ ]:
from IPython.display import display
from tfm.data import project_root
from tfm.m1_profiles import run_m1_profiles

diagnostico_m1 = run_m1_profiles(project_root())
display(diagnostico_m1['fits'].rename(columns={'strategy': 'Regla', 'test_year': 'Modelo de',
    'context_start': 'Primer mes del gráfico', 'context_end': 'Último mes del gráfico',
    'training_examples': 'Señales del entrenamiento', 'context_examples': 'Señales del periodo mostrado',
    'checked_predictions': 'Predicciones de prueba comprobadas'}))

## 36. Leer los cambios de pendiente

Cada fila muestra los modelos ajustados antes del año indicado. Cada cifra bajo la curva cuenta cuántas señales sostienen ese tramo. Un tramo gris tiene menos de ocho señales. La línea se interrumpe si el tramo gris separa un extremo del centro. Busca cambios de pendiente que aparezcan en más de una fila con señales suficientes. Una puntuación mayor indica una probabilidad estimada mayor de superar al efectivo.

In [ ]:
import matplotlib.pyplot as plt
from tfm.m1_profiles import MIN_EXAMPLES, MODELS, YEARS
from tfm.walk_forward import SIGNAL_COLUMNS

formas = diagnostico_m1['curves']
tramos = diagnostico_m1['intervals']
nombres_modelo = {'L2_M1': 'Logística', 'RF_M1': 'Random forest', 'GB_M1': 'Gradient boosting'}
colores_m1 = {'L2_M1': '#303030', 'RF_M1': '#147d70', 'GB_M1': '#753392'}
titulos_m1 = ['Retorno del último mes', 'Momentum 12–1']

for regla, titulo in [('momentum', 'Momentum'), ('reversal', 'Reversión')]:
    fig, axes = plt.subplots(len(YEARS), len(SIGNAL_COLUMNS), figsize=(11, 9),
                             sharex='col', sharey=True, layout='constrained')
    fig.get_layout_engine().set(rect=(0, 0, 1, .92))
    fig.text(.5, .985, f'{titulo} · Cambios en la puntuación a partir del corte central',
             ha='center', va='top', fontsize=14)
    for i, anio in enumerate(YEARS):
        for j, variable in enumerate(SIGNAL_COLUMNS):
            ax = axes[i, j]
            vista = formas.loc[formas.strategy.eq(regla) & formas.test_year.eq(anio) & formas.feature.eq(variable)]
            conteos = tramos.loc[tramos.strategy.eq(regla) & tramos.test_year.eq(anio)
                                  & tramos.feature.eq(variable) & tramos.model.eq('L2_M1')]
            for fila in conteos.itertuples():
                if fila.examples < MIN_EXAMPLES:
                    ax.axvspan(fila.lower, fila.upper, color='#eeeeee', zorder=0)
                ax.text((fila.lower + fila.upper)/2, -.18 if i == len(YEARS) - 1 else -.12, f'n={fila.examples}',
                        transform=ax.get_xaxis_transform(), ha='center', va='top',
                        fontsize=8, color='#555555')
            for modelo in MODELS:
                curva = vista.loc[vista.model.eq(modelo)]
                ax.plot(curva.value, curva.effect, marker='o', markersize=3, color=colores_m1[modelo],
                        linestyle='--' if modelo == 'L2_M1' else '-', label=nombres_modelo[modelo])
            ax.axhline(0, color='#bbbbbb', linewidth=.8, zorder=0)
            ax.grid(axis='y', alpha=.18)
            if i == 0:
                ax.set_title(titulos_m1[j])
            if j == 0:
                ax.set_ylabel(f'{anio}\nCambio en s')
            if i == len(YEARS) - 1:
                ax.set_xlabel('Valor normalizado', labelpad=24)
    lineas, etiquetas = axes[0, 0].get_legend_handles_labels()
    fig.legend(lineas, etiquetas, loc='upper center', bbox_to_anchor=(.5, .959), ncols=3, frameon=False)
    plt.show()

## 37. Un EBM con una curva para cada entrada

Ajustamos **EBM_M1** con retorno del último mes y momentum 12–1. La puntuación es intercepto más los puntos de las dos curvas. La sigmoide convierte esa suma en la probabilidad de que la operación supere al efectivo. Conservamos el calendario de 1950–1989 y la purga del diciembre anterior. Fijamos un máximo de 16 intervalos por entrada, contando los reservados, y 300 rondas para estudiar una forma sencilla con datos mensuales.

Ejecuta desde aquí con las predicciones de boosting de las secciones 32–34 guardadas y el extra `models` actualizado. [Protocolo EBM](../docs/ebm_m1.md)

In [ ]:
import pandas as pd
from IPython.display import display
from tfm.data import project_root, save_result
from tfm.ebm import EBM_PARAMETERS, run_ebm

ROOT = project_root()
display(pd.DataFrame(EBM_PARAMETERS.items(), columns=['Parámetro', 'Valor']))

def progreso_ebm(hechos, total):
    if hechos % 10 == 0 or hechos == total:
        print(f'Años evaluados: {hechos}/{total}')

tablas_ebm, informe_ebm = run_ebm(ROOT, progress=progreso_ebm)
metricas_ebm = tablas_ebm['ebm_walk_forward_metrics']
display(metricas_ebm.loc[metricas_ebm.aggregation.eq('pooled'),
    ['strategy', 'model', 'examples', 'auc', 'log_loss']].round(5))

comparaciones_ebm = tablas_ebm['ebm_walk_forward_comparison']
resumen_ebm = comparaciones_ebm.loc[comparaciones_ebm.aggregation.eq('pooled')].merge(
    pd.DataFrame(informe_ebm['stability']),
    on=['reference_model', 'candidate_model', 'strategy'], validate='one_to_one')
columnas_ebm = {'strategy': 'Regla', 'reference_model': 'Referencia', 'auc_improvement': 'Mejora AUC',
    'log_loss_improvement': 'Mejora log-loss', 'auc_improved_years': 'Años que mejora',
    'auc_worsened_years': 'Años que empeora', 'auc_tied_years': 'Empates',
    'auc_unavailable_years': 'Años sin AUC', 'predictive_support': 'Cumple criterio predictivo'}
display(resumen_ebm[list(columnas_ebm)].rename(columns=columnas_ebm).round(5))

## 38. Qué curva aprende de cada entrada

Cada línea corresponde al modelo disponible al comenzar el año indicado. El eje vertical muestra los puntos que la entrada aporta a la puntuación, y el horizontal contiene los valores normalizados en la zona central estudiada en la sección 36. La sombra marca el tramo del retorno del último mes donde observamos el cambio de pendiente en bosque y boosting.

La tabla muestra cuánto cambia la puntuación en el tramo sombreado y en el inmediatamente anterior. El recuento usa las mismas señales recientes del diagnóstico de M1. La pendiente divide el cambio por la anchura del tramo y permite comparar intervalos de distinto tamaño.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tfm.walk_forward import SIGNAL_COLUMNS

curvas_ebm = tablas_ebm['ebm_m1_curves']
efectos_ebm = tablas_ebm['ebm_m1_local_effects']
anios_ebm = sorted(curvas_ebm.test_year.unique())
colores_ebm = ['#303030', '#276b91', '#147d70', '#753392']
nombres_ebm = ['Retorno del último mes', 'Momentum 12–1']
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex='col', layout='constrained')
fig.get_layout_engine().set(rect=(0, 0, 1, .90))
fig.text(.5, .98, 'EBM · Puntos que aporta cada entrada', ha='center', va='top', fontsize=15)
for i, (regla, titulo) in enumerate([('momentum', 'Momentum'), ('reversal', 'Reversión')]):
    for j, variable in enumerate(SIGNAL_COLUMNS):
        ax = axes[i, j]
        localidad = efectos_ebm.loc[efectos_ebm.feature.eq(variable) & efectos_ebm.strategy.eq(regla)]
        cortes = localidad.loc[localidad.test_year.eq(anios_ebm[0])]
        if j == 0:
            ax.axvspan(cortes.lower.iloc[2], cortes.upper.iloc[2], color='#e1eee8', zorder=0)
        for anio, color in zip(anios_ebm, colores_ebm, strict=True):
            curva = curvas_ebm.loc[curvas_ebm.strategy.eq(regla) & curvas_ebm.feature.eq(variable)
                                   & curvas_ebm.test_year.eq(anio)]
            bordes = np.r_[curva.lower.iloc[0], curva.upper.to_numpy()]
            ax.stairs(curva.contribution, bordes, baseline=None, label=str(anio), color=color, linewidth=1.5)
        ax.axhline(0, color='#999999', linewidth=.7)
        ax.set_xlim(cortes.lower.min(), cortes.upper.max())
        ax.grid(axis='y', alpha=.2)
        if i == 0:
            ax.set_title(nombres_ebm[j])
        if j == 0:
            ax.set_ylabel(titulo + ' · Puntos en s')
        if i == 1:
            ax.set_xlabel('Valor normalizado')
lineas, etiquetas = axes[0, 0].get_legend_handles_labels()
fig.legend(lineas, etiquetas, loc='upper center', bbox_to_anchor=(.5, .943), ncols=4, frameon=False)
plt.show()

vista = efectos_ebm.loc[efectos_ebm.feature.eq(SIGNAL_COLUMNS[0])].copy()
vista['tramo'] = vista.groupby(['strategy', 'test_year']).cumcount().map({1: 'Anterior', 2: 'Sombreado'})
nombres = {'strategy': 'Regla', 'test_year': 'Modelo de', 'tramo': 'Tramo', 'examples': 'Señales',
    'score_change': 'Cambio en s', 'slope': 'Pendiente'}
display(vista.loc[vista.tramo.notna(), list(nombres)].rename(columns=nombres).round(3))

## 39. Guardar probabilidades, métricas y curvas

Estas tablas permiten recuperar las predicciones de cada mes y las aportaciones de los dos términos. El historial cuenta una configuración de EBM por regla.

In [ ]:
guardados_ebm = []
for nombre, tabla in tablas_ebm.items():
    guardado = save_result(ROOT, nombre, tabla, informe_ebm)
    guardados_ebm.append({'Tabla': nombre + '.parquet', 'Filas': guardado['rows']})
display(pd.DataFrame(guardados_ebm))
print('Historial:', informe_ebm['trial_log'])

## 40. ¿Mejoró la cuenta de reversión con el bosque M1?

Comparamos RF_M1 con efectivo, reversión original, veto bajista, ajuste por volatilidad y logística M1. La cuenta nueva acepta las señales con probabilidad superior a 0,5. Conserva su dirección y, en los demás meses, mantiene efectivo. Todas las cuentas parten de 100, abarcan enero de 1950 a diciembre de 1989 e incluyen los mismos costes.

Ejecuta desde aquí con las referencias económicas de las secciones 19–22 y las predicciones del bosque de las secciones 23–25 ya guardadas. [Protocolo](../docs/evaluacion_economica_bosque.md)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tfm.data import project_root, save_result
from tfm.forest_economics import ACCOUNT, REPORT_LABELS, run_forest_economics

ROOT = project_root()
tablas_economicas_bosque, informe_economico_bosque = run_forest_economics(ROOT)
decisiones_bosque = tablas_economicas_bosque['forest_economic_decisions']
cuentas_bosque = tablas_economicas_bosque['forest_economic_simulations']
metricas_economicas_bosque = tablas_economicas_bosque['forest_economic_metrics']
activas = decisiones_bosque.original_signal.ne(0).sum()
largas = decisiones_bosque.filtered_signal.eq(1).sum()
cortas = decisiones_bosque.filtered_signal.eq(-1).sum()
print(f'RF_M1 siguió {largas + cortas} de {activas} señales activas ({largas} largas y {cortas} cortas).')

columnas = {'label': 'Cuenta', 'accepted_signals': 'Meses con posición', 'capital_final': 'Capital final',
    'capital_difference_vs_cash': 'Diferencia frente al efectivo', 'annualized_return': 'Rentabilidad anualizada (%)',
    'sharpe': 'Sharpe', 'max_drawdown': 'Caída máxima (%)',
    'transaction_cost_total': 'Coste de negociación', 'borrow_cost_total': 'Coste de préstamo'}
vista_bosque = metricas_economicas_bosque[list(columnas)].copy()
vista_bosque[['annualized_return', 'max_drawdown']] *= 100
display(vista_bosque.rename(columns=columnas).round(3))

fig, ax = plt.subplots(figsize=(10, 5), layout='constrained')
estilos = {'cash': ('#374450', ':'), 'reversal': ('#986747', '--'),
    'reversal_bear': ('#8f8569', '-.'), 'reversal_volatility': ('#856391', '--'),
    'reversal_m1': ('#326da0', '-'), ACCOUNT: ('#087d64', '-')}
for nombre, etiqueta in REPORT_LABELS.items():
    cuenta = cuentas_bosque.loc[cuentas_bosque.strategy.eq(nombre)]
    meses = pd.PeriodIndex(cuenta.period, freq='M').to_timestamp(how='end')
    fechas = meses.insert(0, meses[0].to_period('M').start_time)
    capital = np.r_[cuenta.capital_start.iloc[0], cuenta.capital_end]
    color, linea = estilos[nombre]
    ax.plot(fechas, capital, label=etiqueta, color=color, linestyle=linea,
            linewidth=2.4 if nombre == ACCOUNT else 1.35)
ax.set_xlim(pd.Timestamp('1950-01-01'), pd.Period('1989-12', freq='M').end_time)
ax.set(title='Reversión · Capital después de costes, 1950–1989', ylabel='Capital (inicio = 100)', xlabel='Año')
ax.grid(axis='y', alpha=.2)
ax.legend(loc='upper left', ncols=2, frameon=True)
plt.show()

## 41. Guardar la cuenta y sus decisiones

La diferencia frente al efectivo se mide en unidades de la cuenta. El Sharpe usa el exceso mensual sobre el efectivo. La cuenta de efectivo aparece sin Sharpe porque su exceso respecto a sí misma es siempre cero.

In [ ]:
guardados_bosque = []
for nombre, tabla in tablas_economicas_bosque.items():
    guardado = save_result(ROOT, nombre, tabla, informe_economico_bosque)
    guardados_bosque.append({'Tabla': nombre + '.parquet', 'Filas': guardado['rows']})
display(pd.DataFrame(guardados_bosque))

## 42. ¿Las dos entradas de M1 aportan información para momentum?

Comparamos M1 con sus fechas reales frente a 642 versiones que desplazan juntas sus dos entradas por el calendario. Las fechas y etiquetas de las operaciones se conservan. Volvemos a ajustar cada versión en las mismas quince particiones CPCV y promediamos las cinco probabilidades por señal.

Cada resultado mide cuánto reduce la log-loss respecto a M0. M0 usa la frecuencia de éxitos del entrenamiento de cada partición. Para pasar el criterio, M1 debe mejorar la log-loss de M0, alcanzar un AUC de al menos 0,54 y superar el percentil 95 de las mejoras de las sombras. Ejecuta desde aquí con las salidas de CPCV de las secciones 16–18 ya guardadas. [Protocolo](../docs/sombras_momentum_m1.md)

In [ ]:
from IPython.display import display
from tfm.data import project_root, save_result
from tfm.momentum_shadows import run_momentum_shadows

ROOT = project_root()

def progreso_sombras(hechas, total):
    print(f'Sombras completadas: {hechas}/{total}')

tablas_sombras_m1, informe_sombras_m1 = run_momentum_shadows(ROOT, progress=progreso_sombras)
sombras_m1 = tablas_sombras_m1['momentum_m1_shadows']
comparacion_sombras_m1 = tablas_sombras_m1['momentum_m1_shadow_comparison']
columnas = {'original_auc': 'AUC real', 'original_log_loss': 'Log-loss M1', 'naive_log_loss': 'Log-loss M0',
    'original_improvement': 'Mejora real frente a M0', 'improvement_threshold_95': 'Percentil 95 de sombras',
    'completed_shadows': 'Sombras completas', 'meets_operational_criterion': 'Cumple criterio'}
display(comparacion_sombras_m1[list(columnas)].rename(columns=columnas).round(5))

## 43. Leer y guardar la comparación con sombras

Las barras muestran cuántas sombras obtuvieron cada rango de mejora frente a M0. Los valores negativos corresponden a sombras que empeoraron la log-loss. La línea de M1 real se compara con la del percentil 95. Este contraste mide predicción durante el desarrollo. La utilidad económica se estudia con las cuentas continuas.

In [ ]:
import matplotlib.pyplot as plt

comparacion = comparacion_sombras_m1.iloc[0]
fig, ax = plt.subplots(figsize=(9, 4.5), layout='constrained')
if comparacion.comparison_available:
    ax.hist(sombras_m1.log_loss_improvement, bins=25, color='#91b7aa', edgecolor='white',
            label='Sombras de M1')
    ax.axvline(comparacion.improvement_threshold_95, color='#70518c', linestyle='--',
               label='Percentil 95 de las sombras')
    ax.axvline(comparacion.original_improvement, color='#113f70', linewidth=2, label='M1 con fechas reales')
    ax.axvline(0, color='#777777', linestyle=':', linewidth=1, label='Igual a M0')
    ax.legend()
else:
    ax.text(.5, .5, 'La comparación necesita todas las sombras completas.',
            transform=ax.transAxes, ha='center')
ax.set(title='Momentum M1 · Comparación predictiva con sombras',
       xlabel='Mejora de log-loss frente a M0', ylabel='Número de sombras')
plt.show()

guardados_sombras_m1 = []
for nombre, tabla in tablas_sombras_m1.items():
    guardado = save_result(ROOT, nombre, tabla, informe_sombras_m1)
    guardados_sombras_m1.append(f'{nombre}.parquet ({guardado["rows"]} filas)')
print('Tablas guardadas:', ', '.join(guardados_sombras_m1))

## 44. ¿Cuánta incertidumbre tiene la mejora de Sharpe de momentum M1?

Comparamos momentum M1 con momentum sin filtro, con veto bajista y con ajuste por volatilidad. Usamos los mismos 480 meses de 1950–1989, con los retornos después de costes y descontando el efectivo de cada mes. Esta sección usa las cuentas guardadas en 19–22 y se puede ejecutar en un kernel nuevo.

Aplicamos el bootstrap circular studentizado de Ledoit y Wolf con una adaptación de longitud fija. Formamos 5.000 historiales alternativos concatenando bloques de doce meses. Cada bloque conserva juntas las dos cuentas para que sigan viviendo el mismo mercado. El error estándar de la serie original usa una estimación HAC de Bartlett con once retardos. Los intervalos y valores p son bilaterales y se leen por comparación. Un intervalo que incluye cero indica que, con este contraste, seguimos teniendo incertidumbre sobre el signo de la diferencia.

[Protocolo y referencia](../docs/contraste_economico.md)

In [ ]:
from IPython.display import display
from tfm.data import project_root
from tfm.economic_validation import run_economic_validation

contraste_economico = run_economic_validation(project_root())
display(contraste_economico['comparisons'].rename(columns={
    'reference': 'Comparador',
    'months': 'Meses',
    'sharpe_candidate': 'Sharpe de M1',
    'sharpe_reference': 'Sharpe del comparador',
    'sharpe_difference': 'Diferencia de Sharpe',
    'ci_low': 'Límite inferior (95 %)',
    'ci_high': 'Límite superior (95 %)',
    'p_value': 'Valor p bilateral',
}).round(4))

## 45. ¿En qué décadas se aprecia la diferencia?

Tomamos los retornos de las mismas cuentas continuas y calculamos su Sharpe en cuatro décadas de 120 meses. Así vemos si la diferencia aparece en varios tramos. La tabla es descriptiva y conserva los costes originales, sin reinicios de cuenta ni liquidaciones adicionales al terminar cada década.

In [ ]:
diferencia_decadas = contraste_economico['decades'].pivot(
    index='decade', columns='reference', values='sharpe_difference'
)
diferencia_decadas.index = [f'{start}–{start + 9}' for start in diferencia_decadas.index]
display(diferencia_decadas.rename_axis('Década').rename_axis(
    'Diferencia de Sharpe de M1 frente a cada comparador', axis=1
).round(4))

## 46. ¿Qué series tenemos para calcular DSR y PBO?

El registro distingue cada variante y regla. Sus repeticiones, las particiones y las sombras sirven para comprobarla. Una cuenta disponible contiene 480 retornos netos emparejados y se vincula al intento que la produjo. Las cuentas de reversión con el bosque aparecen si se han guardado en 40–41.

Este inventario describe las cuentas evaluadas individualmente. Las secciones 47–48 preparan la matriz común. La logística de diez variables probada inicialmente con CPCV corresponde a la misma especificación de entradas que el último escalón por bloques. Sus resultados se conservan con sus dos identificadores. M0 y los comparadores sencillos sirven como referencias. Las predicciones, las sombras y las curvas explicativas tienen sus propios registros y usos.

In [ ]:
inventario_economico = contraste_economico['inventory']
display(inventario_economico.groupby(['rule', 'next_step']).size().unstack(fill_value=0).rename_axis('Regla'))
display(inventario_economico[['rule', 'candidate', 'attempts', 'account', 'next_step', 'same_inputs']].rename(
    columns={'rule': 'Regla', 'candidate': 'Variante', 'attempts': 'Intentos',
             'account': 'Cuenta', 'next_step': 'Estado de la serie', 'same_inputs': 'Entradas coincidentes'}
))

## 47. Qué variantes entran en la comparación económica

Construimos cuentas para las 16 variantes cronológicas de cada regla. Incluimos la logística M1–M3, las tres retiradas de variables de M2, los cuatro bloques sucesivos, las tres variantes de random forest, las dos de boosting y EBM. Los comparadores fijos quedan identificados como referencias. La logística de diez variables está representada por el último escalón por bloques. Clustered MDA se estudió con un selector interno de CPCV y su resultado permanece en el notebook 13.

Hay una fila por cuenta y mes, de enero de 1950 a diciembre de 1989. La cuenta conserva la dirección original de la señal cuando la probabilidad supera 0,5 y pasa a efectivo en los demás casos. Usamos las cuentas de 19–22 y las predicciones guardadas de 13–15, 23–25, 32–34 y 37–39 de este notebook. Se reproducen en memoria los cuatro bloques de 30–31 con sus listas originales. El periodo desde 1990 queda reservado.

[Protocolo y variantes](../docs/series_dsr_pbo.md)

In [ ]:
from IPython.display import display
from tfm.data import project_root, save_result
from tfm.selection_returns import RESULT_NAME as SERIES_DSR_PBO, run_selection_returns

matriz_economica, informe_matriz = run_selection_returns(project_root())
cobertura_matriz = matriz_economica.groupby(['rule', 'role'], as_index=False).agg(
    Cuentas=('policy', 'nunique'), Meses_por_cuenta=('period', 'nunique'), Filas=('period', 'size')
)
cobertura_matriz['role'] = cobertura_matriz['role'].map({'candidate': 'Filtro', 'reference': 'Referencia'})
display(cobertura_matriz.rename(columns={'rule': 'Regla', 'role': 'Tipo', 'Meses_por_cuenta': 'Meses por cuenta'}))

## 48. Guardar los retornos netos sincronizados

Guardamos el retorno neto y el del efectivo de cada mes, junto con las variantes y el origen de las predicciones. El siguiente paso utilizará esta matriz para calcular DSR y PBO dentro del periodo de desarrollo.

In [ ]:
salida_matriz = save_result(project_root(), SERIES_DSR_PBO, matriz_economica, informe_matriz)
print('Tabla guardada:', salida_matriz['path'])
print('Filas:', salida_matriz['rows'])

## 49. DSR · ¿Qué Sharpe esperaríamos tras probar varios filtros?

Usamos la matriz de 1950–1989 guardada en 47–48. Dentro de cada regla, las 16 variantes se comparan sobre el exceso neto respecto al efectivo. La correlación media entre sus series permite estimar cuántas pruebas equivalentes serían independientes. Se redondea hacia arriba para calcular la referencia y se muestra también el caso de tratar las 16 como independientes.

La referencia del DSR es el mejor Sharpe que cabría esperar por selección si los filtros tuvieran Sharpe nulo. El DSR incorpora la dispersión de los Sharpes, los meses disponibles, la asimetría y la curtosis de los retornos. Se muestran los Sharpes en escala anualizada. Para comparar M1 directamente con las reglas sencillas se utiliza el contraste de las secciones 44–45.

[Protocolo y referencias](../docs/dsr_pbo.md)

In [ ]:
from IPython.display import display
from tfm.data import project_root, save_result
from tfm.overfitting import RESULT_NAMES as OVERFITTING_RESULTS, run_overfitting

tablas_sobreajuste, informe_sobreajuste = run_overfitting(project_root())
dsr = tablas_sobreajuste[OVERFITTING_RESULTS[0]]
campos_pruebas = ['rule', 'trials', 'mean_return_correlation', 'effective_trials', 'trials_used',
                  'benchmark_annualized', 'independent_benchmark_annualized']
display(dsr[campos_pruebas].drop_duplicates().rename(columns={
    'rule': 'Regla', 'trials': 'Filtros', 'mean_return_correlation': 'Correlación media',
    'effective_trials': 'Pruebas efectivas estimadas', 'trials_used': 'Pruebas usadas',
    'benchmark_annualized': 'Sharpe de referencia',
    'independent_benchmark_annualized': 'Referencia con 16 independientes',
}).round(4))
display(dsr[['rule', 'policy', 'sharpe_annualized', 'dsr', 'dsr_all_independent']].rename(columns={
    'rule': 'Regla', 'policy': 'Filtro', 'sharpe_annualized': 'Sharpe anualizado',
    'dsr': 'DSR', 'dsr_all_independent': 'DSR con 16 independientes',
}).round(6))

## 50. PBO · ¿Qué pasa con el ganador al cambiar de meses?

Dividimos los 480 meses en 16 bloques consecutivos de 30 meses. Para cada una de las 12.870 formas de elegir ocho bloques, buscamos el filtro con el mayor Sharpe. En los otros ocho miramos en qué posición queda entre los mismos 16 filtros. Ya tenemos sus retornos calculados con el recorrido cronológico. Aquí cambiamos los meses que usamos para comparar y ordenar las cuentas.

El PBO es la proporción de veces que el ganador queda por debajo de la mediana en la mitad complementaria. Un filtro puede conservar una buena posición relativa y aun así tener un Sharpe negativo. Por eso mostramos ese porcentaje por separado. Las combinaciones comparten meses y el resultado se circunscribe al desarrollo.

In [ ]:
pbo = tablas_sobreajuste[OVERFITTING_RESULTS[1]]
particiones_pbo = tablas_sobreajuste[OVERFITTING_RESULTS[2]]
display(pbo.assign(pbo_pct=pbo.pbo * 100, negative_pct=pbo.negative_sharpe_fraction * 100)[
    ['rule', 'splits', 'below_median_splits', 'at_median_splits', 'selection_tie_splits',
     'pbo_pct', 'negative_pct']
].rename(columns={
    'rule': 'Regla', 'splits': 'Combinaciones', 'below_median_splits': 'Por debajo de la mediana',
    'at_median_splits': 'En la mediana', 'selection_tie_splits': 'Empates al elegir',
    'pbo_pct': 'PBO (%)', 'negative_pct': 'Sharpe negativo en el complemento (%)',
}).round(2))
display(particiones_pbo.loc[particiones_pbo.split_id.eq(1), [
    'rule', 'selection_blocks', 'selected_policy', 'sharpe_selection',
    'sharpe_complement', 'complement_rank'
]].rename(columns={
    'rule': 'Regla', 'selection_blocks': 'Bloques usados para elegir', 'selected_policy': 'Ganador',
    'sharpe_selection': 'Sharpe al elegir', 'sharpe_complement': 'Sharpe en el complemento',
    'complement_rank': 'Rango complementario',
}).round(4))

## 51. Leer el rango y guardar los controles

En el gráfico, el rango 1 es el peor y el 16 es el mejor. Cada barra muestra con qué frecuencia el ganador de una mitad obtuvo ese rango en los meses restantes. A la izquierda del punto medio están los casos que forman el PBO. El DSR mide la evidencia del Sharpe frente a su referencia ajustada por selección. El PBO describe la estabilidad de elegir por Sharpe.

Se guardan el DSR de cada filtro, el PBO por regla y el detalle de cada combinación. El periodo desde 1990 permanece reservado.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True, layout='constrained')
for ax, (rule, title) in zip(axes, [('momentum', 'Momentum'), ('reversal', 'Reversión')]):
    rows = particiones_pbo.loc[particiones_pbo.rule.eq(rule)]
    ranks = rows.complement_rank.value_counts(normalize=True).sort_index()
    midpoint = (int(pbo.loc[pbo.rule.eq(rule), 'trials'].iloc[0]) + 1) / 2
    colors = ['#a44912' if rank < midpoint else '#005a57' for rank in ranks.index]
    ax.bar(ranks.index, ranks, color=colors, width=.44)
    ax.axvline(midpoint, linestyle='--', color='#555555', linewidth=1)
    probability = pbo.loc[pbo.rule.eq(rule), 'pbo'].iloc[0]
    ax.set(title=f'{title} · PBO {probability:.1%}', xlabel='Rango en el complemento',
           xlim=(.5, 16.5), xticks=[1, 4, 8, 12, 16])
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.grid(axis='y', alpha=.25)
    ax.set_axisbelow(True)
axes[0].set_ylabel('Combinaciones')
plt.show()

guardados_sobreajuste = []
for nombre, tabla in tablas_sobreajuste.items():
    salida = save_result(project_root(), nombre, tabla, informe_sobreajuste)
    guardados_sobreajuste.append({'Tabla': salida['path'], 'Filas': salida['rows']})
display(guardados_sobreajuste)